In [3]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ==========================================
# 1. CONFIGURATION DES CHEMINS ET PARAMÈTRES
# ==========================================
BASE_DIR = ".."
# On utilise le chemin exact qu'on a validé ensemble
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw", "plantvillage dataset", "color")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 5

# ==========================================
# 2. PRÉPARATION DES DONNÉES (FLUX)
# ==========================================
# On normalise les pixels et on sépare 20% pour la validation
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

print("⏳ Indexation des images en cours... (Patientez)")

train_generator = datagen.flow_from_directory(
    RAW_DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    RAW_DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# ==========================================
# 3. CONSTRUCTION DU MODÈLE (ARCHITECTURE)
# ==========================================
print("🧠 Configuration de l'IA (MobileNetV2)...")

base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False # On ne ré-entraîne pas la base, seulement la fin

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(train_generator.num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# ==========================================
# 4. LANCEMENT DE L'ENTRAÎNEMENT
# ==========================================
print("🚀 Lancement de l'entraînement...")

history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE
)

# ==========================================
# 5. SAUVEGARDE
# ==========================================
if not os.path.exists('../models'):
    os.makedirs('../models')

model.save('../models/agri_model_v1.h5')
print("\n✅ Terminé ! Le modèle est sauvegardé sous 'models/agri_model_v1.h5'")

⏳ Indexation des images en cours... (Patientez)
Found 43456 images belonging to 38 classes.
Found 10849 images belonging to 38 classes.
🧠 Configuration de l'IA (MobileNetV2)...
🚀 Lancement de l'entraînement...
Epoch 1/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 1837s 1s/step - accuracy: 0.8701 - loss: 0.4741 - val_accuracy: 0.9348 - val_loss: 0.2138
Epoch 2/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 1690s 1s/step - accuracy: 0.9375 - loss: 0.1988 - val_accuracy: 0.9448 - val_loss: 0.1702
Epoch 3/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 1541s 1s/step - accuracy: 0.9496 - loss: 0.1588 - val_accuracy: 0.9486 - val_loss: 0.1554
Epoch 4/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 1525s 1s/step - accuracy: 0.9559 - loss: 0.1357 - val_accuracy: 0.9551 - val_loss: 0.1369
Epoch 5/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 1622s 1s/step - accuracy: 0.9586 - loss: 0.1262 - val_accuracy: 0.9503 - val_loss: 0.1511



✅ Terminé ! Le modèle est sauvegardé sous 'models/agri_model_v1.h5'
